In [1]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack

from catboost import CatBoostClassifier

In [2]:
root_path = "/home/stefan/ioai-prep/kits/autor_versuri"
seed = 423

# Data

In [3]:
train_df = pd.read_csv(os.path.join(root_path, "train.csv"))
test_df = pd.read_csv(os.path.join(root_path, "test.csv"))

le = LabelEncoder()
y = le.fit_transform(train_df["Autor"])
X_train_text = train_df["Versuri"].values
X_test_text = test_df["Versuri"].values

In [4]:
class StyleFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            lines = text.strip().split("\n")
            words = re.findall(r"\b\w+\b", text.lower())
            chars = re.sub(r"\s", "", text)

            f = {
                "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
                "avg_line_len": np.mean([len(l) for l in lines]) if lines else 0,
                "num_lines": len(lines),
                "vocab_richness": len(set(words)) / len(words) if words else 0,
                "punct_ratio": sum(1 for c in text if c in ".,;:!?-—") / len(text),
                "comma_ratio": text.count(",") / len(text),
                "ellipsis": text.count("...") / len(text) * 100,
                "exclaim_ratio": text.count("!") / len(text),
                "question_ratio": text.count("?") / len(text),
                "dash_ratio": (text.count("-") + text.count("—")) / len(text),
                "uppercase_ratio": (
                    sum(1 for c in chars if c.isupper()) / len(chars) if chars else 0
                ),
                "digit_ratio": sum(1 for c in text if c.isdigit()) / len(text),
                "diacritics": sum(1 for c in text if c in "ăâîșțĂÂÎȘȚ") / len(text),

                "lines_ending_a": sum(
                    1 for l in lines if l.strip().lower().endswith("a")
                ),
                "lines_ending_e": sum(
                    1 for l in lines if l.strip().lower().endswith("e")
                ),
                "lines_ending_i": sum(
                    1 for l in lines if l.strip().lower().endswith("i")
                ),
            }
            features.append(list(f.values()))
        return np.array(features)

In [5]:
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=15000,
    sublinear_tf=True,
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    min_df=2,
)

In [6]:
def extract_line_endings(texts, n_chars=8):
    result = []
    for text in texts:
        lines = text.strip().split("\n")
        endings = " ".join(
            [l.strip()[-n_chars:] for l in lines if len(l.strip()) >= n_chars]
        )
        result.append(endings)
    return result


line_end_train = extract_line_endings(X_train_text)
line_end_test = extract_line_endings(X_test_text)

ending_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    max_features=3000,
)

In [7]:
print("Building features...")
X_char = char_vectorizer.fit_transform(X_train_text)
X_word = word_vectorizer.fit_transform(X_train_text)
X_endings = ending_vectorizer.fit_transform(line_end_train)
X_style = StyleFeatures().fit_transform(X_train_text)

scaler = StandardScaler()
X_style_scaled = scaler.fit_transform(X_style)

X_combined = hstack([X_char, X_word, X_style_scaled])

X_char_test = char_vectorizer.transform(X_test_text)
X_word_test = word_vectorizer.transform(X_test_text)
#X_endings_test = ending_vectorizer.transform(line_end_test)
X_style_test = scaler.transform(StyleFeatures().transform(X_test_text))

X_test_combined = hstack([X_char_test, X_word_test, X_style_test])

print(f"Feature dimensions: {X_combined.shape}")

Building features...
Feature dimensions: (3415, 25016)


# Models

In [8]:
def evaluate(model, cb=-1):
    cv = cross_val_score(model, X_combined, y, cv=(3 if cb == -1 else 2), scoring="accuracy", n_jobs=cb)
    return cv.mean() - cv.std()

In [ ]:
lr = LogisticRegression(C=5, max_iter=2000)
evaluate(lr)

0.7301701981838175

In [ ]:
svc = LinearSVC(C=1.5, max_iter=9000)
evaluate(svc)

0.7618552275871653

In [ ]:
stacking = StackingClassifier(
    estimators=[('lr', lr), ('svc', svc)],
    final_estimator=LogisticRegression(C=1, max_iter=1000),
    cv=3,
    n_jobs=-1,
)

evaluate(svc)

In [ ]:
model = stacking

model.fit(X_combined, y)

NameError: name 'stacking' is not defined

# Submission

In [ ]:
preds = model.predict(X_test_combined)

submission = pd.DataFrame(
    {
        "Id": test_df["Id"],
        "Autor": le.inverse_transform(preds),
    }
)
submission.to_csv(os.path.join(root_path, "submission.csv"), index=False)
print("\nSubmission saved.")


Submission saved.
